# The Mapper Algorithm with KeplerMapper

**Mapper** is a topological method that produces a simplicial complex (graph) summarising the shape of high-dimensional data. This notebook:
1. Explains the Mapper pipeline
2. Applies KeplerMapper to the **UCI Wine** dataset
3. Visualises the resulting graph and interprets clusters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

try:
    import kmapper as km
    HAS_KMAPPER = True
except ImportError:
    HAS_KMAPPER = False
    print('kmapper not installed -- pip install kmapper')

%matplotlib inline

## 1. The Mapper Pipeline

Mapper has four ingredients:
1. **Filter function** $f: X \to \mathbb{R}^d$ -- a lens that reveals structure (e.g., PCA projection, eccentricity)
2. **Cover** of the filter range -- overlapping intervals
3. **Clustering** within each pre-image $f^{-1}(U_i)$
4. **Nerve construction** -- nodes are clusters, edges connect overlapping clusters

The result is a graph (1-skeleton of a simplicial complex) that captures the topological shape of $X$.

In [ ]:
# Prepare data
wine = load_wine()
X = StandardScaler().fit_transform(wine.data)
y = wine.target
print(f"Wine dataset: {X.shape[0]} samples, {X.shape[1]} features, {len(np.unique(y))} classes")

In [ ]:
if HAS_KMAPPER:
    # Initialise Mapper
    mapper = km.KeplerMapper(verbose=1)
    
    # Step 1: Project data using the filter function (2D PCA)
    lens = mapper.fit_transform(X, projection=PCA(n_components=2))
    print(f"Filter output shape: {lens.shape}")

In [ ]:
if HAS_KMAPPER:
    # Step 2-4: Build the Mapper graph
    graph = mapper.map(
        lens, X,
        cover=km.Cover(n_cubes=10, perc_overlap=0.3),
        clusterer=DBSCAN(eps=1.5, min_samples=3)
    )
    
    print(f"Mapper graph: {len(graph['nodes'])} nodes, "
          f"{sum(len(v) for v in graph['links'].values())//2} edges")

In [ ]:
if HAS_KMAPPER:
    # Visualise: colour nodes by average wine class
    html = mapper.visualize(
        graph,
        path_html='mapper_wine.html',
        title='Mapper on Wine Dataset',
        color_values=y,
        color_function_name='Wine class',
        custom_tooltips=np.array([f'Class {c}' for c in y])
    )
    print('Interactive visualisation saved to mapper_wine.html')
    print('Open it in a browser to explore the topological graph.')

In [ ]:
if HAS_KMAPPER:
    # Static matplotlib visualisation using networkx
    try:
        import networkx as nx
        
        G = km.adapter.to_nx(graph)
        pos = nx.spring_layout(G, seed=42)
        
        # Colour each node by the dominant class
        node_colors = []
        node_sizes = []
        for node in G.nodes():
            members = graph['nodes'][node]
            node_sizes.append(len(members) * 10)
            classes = y[members]
            node_colors.append(np.bincount(classes).argmax())
        
        plt.figure(figsize=(10, 8))
        nx.draw_networkx(
            G, pos, node_color=node_colors, node_size=node_sizes,
            cmap='Set1', with_labels=False, edge_color='gray', alpha=0.8
        )
        plt.title('Mapper Graph -- Wine Dataset (coloured by dominant class)')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    except ImportError:
        print('networkx not installed for static plot -- see the HTML visualisation instead.')

## 2. Interpreting the Mapper Graph

- **Connected components** correspond to well-separated groups.
- **Branching** reveals sub-populations or gradients in the data.
- **Loops** indicate cyclic or recurrent patterns.
- The graph provides a **compressed, interpretable view** of high-dimensional structure.

In [ ]:
if HAS_KMAPPER:
    # Analyse node composition
    print("Node analysis (first 10 nodes):")
    print(f"{'Node':<15} {'Size':>5} {'Class 0':>8} {'Class 1':>8} {'Class 2':>8}")
    for i, (node_id, members) in enumerate(graph['nodes'].items()):
        if i >= 10:
            break
        classes = y[members]
        counts = np.bincount(classes, minlength=3)
        print(f"{node_id:<15} {len(members):>5} {counts[0]:>8} {counts[1]:>8} {counts[2]:>8}")

## Key Takeaways

- **Mapper** is a flexible, parameter-driven tool for topological data exploration.
- It produces a graph that compresses high-dimensional datasets into interpretable shape summaries.
- **KeplerMapper** provides both interactive HTML and static visualisations.
- Mapper is widely used in genomics, materials science, and financial data analysis.

For a deeper treatment, see: Singh, Memoli, Carlsson (2007). *Topological Methods for the Analysis of High Dimensional Data Sets and 3D Object Recognition*.